<br>
<br>
<br>

### Preparing Crime Rate Data for Merging

Lets first read in our data from https://files.crimestatistics.vic.gov.au/2026-09/Data_Tables_LGA_Recorded_Offences_Year_Ending_June_2026.xlsx which details crime rate information of each Local Goverment Area (LGA) from June 2016 to June 2026.

In [14]:
import pandas as pd

FILE = "../external_data/Data_Tables_LGA_Recorded_Offences_Year_Ending_June_2026.xlsx"

t1 = pd.read_excel(FILE, sheet_name="Table 01")
t1 = t1.drop(columns=["Police Region"]) # Drop police regions; not necessary 

print(t1.head())

   Year Year ending Local Government Area  Offence Count  \
0  2026        June               Banyule          10632   
1  2026        June              Brimbank          18704   
2  2026        June               Darebin          16174   
3  2026        June           Hobsons Bay           7785   
4  2026        June                  Hume          23691   

   Rate per 100,000 population  
0                  7978.035302  
1                  9402.732059  
2                  9941.035496  
3                  7959.260566  
4                  8169.137486  


### Cleaning Crime Rate Data

#### Lets Proceed Further with cleaning:
1. Strip leading/trailing spaces from LGA names
2. Drop rows that aren't real councils
3. Handle Nulls
4. Duplicates: each Year + LGA should appear once
5. Data types

##### Step 1: Strip leading/trailing spaces from LGA names

In [15]:
LGA_COL = "Local Government Area"
NON_LGAS = [
    "Total",
    "Justice Institutions and Immigration Facilities",
    "Unincorporated Vic"
]

has_spaces = t1[LGA_COL] != t1[LGA_COL].str.strip()
print("\nStep 1: LGA names with extra spaces:", has_spaces.sum())
print("Example before:", repr(t1.loc[has_spaces, LGA_COL].iloc[0])
      if has_spaces.any() else "none")

t1[LGA_COL] = t1[LGA_COL].str.strip()

has_spaces = t1[LGA_COL] != t1[LGA_COL].str.strip()
print("After stripping:", has_spaces.sum())


Step 1: LGA names with extra spaces: 810
Example before: ' Banyule'
After stripping: 0


##### Step 2: Drop rows that aren't real councils

In [16]:
is_non_lga = t1[LGA_COL].isin(NON_LGAS)
print("\nStep 2: Non-council rows found:", is_non_lga.sum())
print(t1.loc[is_non_lga, LGA_COL].value_counts())

print("Rows before dropping:", len(t1))
t1 = t1[~is_non_lga].reset_index(drop=True)
print("Rows after dropping:", len(t1))
print("Unique LGAs remaining:", t1[LGA_COL].nunique())



Step 2: Non-council rows found: 70
Local Government Area
Total                                              60
Justice Institutions and Immigration Facilities    10
Name: count, dtype: int64
Rows before dropping: 870
Rows after dropping: 800
Unique LGAs remaining: 80


##### Step 3: Check for missing values

In [17]:
missing = t1.isna().sum()
print("\nStep 3: Missing values per column:")
print(missing)
print("Total missing:", missing.sum())


Step 3: Missing values per column:
Year                            0
Year ending                     0
Local Government Area           0
Offence Count                   0
Rate per 100,000 population    10
dtype: int64
Total missing: 10


##### Step 4: Check for duplicates

In [18]:
full_dupes = t1.duplicated().sum()
key_dupes = t1.duplicated(["Year", LGA_COL]).sum()
print("\nStep 4: Fully identical rows:", full_dupes)
print("Duplicate Year + LGA pairs:", key_dupes)


Step 4: Fully identical rows: 0
Duplicate Year + LGA pairs: 0


##### Step 5: Check Data Types

In [19]:
print("\nStep 5: Data types:")
print(t1.dtypes)
print("\nYears covered:", sorted(t1["Year"].unique()))


Step 5: Data types:
Year                             int64
Year ending                     object
Local Government Area           object
Offence Count                    int64
Rate per 100,000 population    float64
dtype: object

Years covered: [np.int64(2017), np.int64(2018), np.int64(2019), np.int64(2020), np.int64(2021), np.int64(2022), np.int64(2023), np.int64(2024), np.int64(2025), np.int64(2026)]


Secondly, lets read in our data from https://www.crimestatistics.vic.gov.au/download-data-15 (Data tables - Recorded offences (XLSX, 510.61 KB)) which details crime rate information of each Local Goverment Area (LGA) from Jul 2011 to June 2016.

In [20]:
import re

FILE_TWO = "../external_data/Offences data table year ending 30 June 2016.xlsx"
RATE_COL = "Rate per 100,000 population"

print(pd.ExcelFile(FILE_TWO).sheet_names)  # Find the Table 8 sheet name
raw = pd.read_excel(FILE_TWO, sheet_name="Table 8", header=None)

# Find the header row ("Jul 2011 - Jun 2012" ...) and the LGA name column
text = raw.astype(str)
header_row = text.apply(
    lambda row: row.str.contains("Jul 2011").any(), axis=1
).idxmax()
name_col = text.apply(lambda col: col.str.strip().eq("Banyule").any()).idxmax()

# Map each year column to its end year, e.g. "Jul 2011 - Jun 2012" -> 2012
year_cols = {}
for col, value in raw.loc[header_row].items():
    match = re.search(r"Jun\s*(\d{4})", str(value))
    if match:
        year_cols[col] = int(match.group(1))

# Keep the LGA names and the year columns below the header
old = raw.loc[header_row + 1:, [name_col, *year_cols]]
old = old.rename(columns={name_col: LGA_COL, **year_cols})
old[LGA_COL] = old[LGA_COL].astype(str).str.strip()

# Region headings have no numbers, so drop rows with no rates
year_list = list(year_cols.values())
old[year_list] = old[year_list].apply(pd.to_numeric, errors="coerce")
old = old.dropna(subset=year_list, how="all")

# Drop subtotals, totals and non-council rows
is_total = old[LGA_COL].str.contains("total", case=False)
old = old[~is_total & ~old[LGA_COL].isin(NON_LGAS)]

# Wide -> long: one row per LGA per year
old = old.melt(id_vars=LGA_COL, var_name="Year", value_name=RATE_COL)
old["Year"] = old["Year"].astype(int)

# Moreland was renamed Merri-bek in 2022
old[LGA_COL] = old[LGA_COL].replace({"Moreland": "Merri-bek"})

print(old.head(10))
print(old[LGA_COL].nunique(), "LGAs,", sorted(old["Year"].unique()))

['Contents', 'Table 1', 'Table 2', 'Table 3', 'Table 4', 'Table 5', 'Table 6', 'Table 7', 'Table 8', 'Table 9', 'Table 10', 'Table 11', 'Table 12']
  Local Government Area  Year  Rate per 100,000 population
0               Banyule  2012                  5829.288026
1              Brimbank  2012                  9008.762039
2               Darebin  2012                  8147.067567
3           Hobsons Bay  2012                  7379.212688
4                  Hume  2012                  8383.791860
5           Maribyrnong  2012                 11816.667968
6             Melbourne  2012                 30312.660077
7                Melton  2012                  7148.734741
8         Moonee Valley  2012                  6638.006220
9             Merri-bek  2012                  7405.350543
79 LGAs, [np.int64(2012), np.int64(2013), np.int64(2014), np.int64(2015), np.int64(2016)]


We will follow all the cleaning steps as done previously.

In [21]:
# ---------------------------------------------------------------
# Step 1: Leftover spaces in LGA names
# ---------------------------------------------------------------
has_spaces = old[LGA_COL] != old[LGA_COL].str.strip()
print("Step 1: LGA names with extra spaces:", has_spaces.sum())

# ---------------------------------------------------------------
# Step 2: Non-council rows remaining
# ---------------------------------------------------------------
print("\nStep 2: Unique LGAs:", old[LGA_COL].nunique())

# ---------------------------------------------------------------
# Step 3: Missing values
# ---------------------------------------------------------------
print("\nStep 3: Missing values per column:")
print(old.isna().sum())
print(old[old[RATE_COL].isna()])

# ---------------------------------------------------------------
# Step 4: Duplicates
# ---------------------------------------------------------------
print("\nStep 4: Fully identical rows:", old.duplicated().sum())
print("Duplicate Year + LGA pairs:",
      old.duplicated(["Year", LGA_COL]).sum())

# ---------------------------------------------------------------
# Step 5: Data types
# ---------------------------------------------------------------
print("\nStep 5: Data types:")
print(old.dtypes)

# ---------------------------------------------------------------
# Step 6: LGA names match the current data
# ---------------------------------------------------------------
old_names = set(old[LGA_COL])
new_names = set(t1[LGA_COL])
print("\nStep 6: In 2016 file but not current:",
      sorted(old_names - new_names))
print("In current but not 2016 file:", sorted(new_names - old_names))

Step 1: LGA names with extra spaces: 0

Step 2: Unique LGAs: 79

Step 3: Missing values per column:
Local Government Area          0
Year                           0
Rate per 100,000 population    0
dtype: int64
Empty DataFrame
Columns: [Local Government Area, Year, Rate per 100,000 population]
Index: []

Step 4: Fully identical rows: 0
Duplicate Year + LGA pairs: 0

Step 5: Data types:
Local Government Area           object
Year                             int64
Rate per 100,000 population    float64
dtype: object

Step 6: In 2016 file but not current: ['Warnambool']
In current but not 2016 file: ['Unincorporated Vic', 'Warrnambool']


In [22]:
print("old:", old.shape)
print("t1: ", t1.shape)

print("\nold columns:", list(old.columns))
print("t1 columns: ", list(t1.columns))

print("\nold years:", sorted(old["Year"].unique()))
print("t1 years: ", sorted(t1["Year"].unique()))

old: (395, 3)
t1:  (800, 5)

old columns: ['Local Government Area', 'Year', 'Rate per 100,000 population']
t1 columns:  ['Year', 'Year ending', 'Local Government Area', 'Offence Count', 'Rate per 100,000 population']

old years: [np.int64(2012), np.int64(2013), np.int64(2014), np.int64(2015), np.int64(2016)]
t1 years:  [np.int64(2017), np.int64(2018), np.int64(2019), np.int64(2020), np.int64(2021), np.int64(2022), np.int64(2023), np.int64(2024), np.int64(2025), np.int64(2026)]


<br>
<br>
<br>

### Final Steps and Saving the Cleaned Dataset

Finally, lets drop the columns we dont need and save the file

##### Lets merge both files together now and save it to disc:

In [23]:
from pathlib import Path

OUTPUT_DIR = Path("../external_data")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# Fix the 2016 file's spelling of Warrnambool
old[LGA_COL] = old[LGA_COL].replace({"Warnambool": "Warrnambool"})

# Combine 2012-2016 with 2017-2026
crime_all = pd.concat(
    [old, t1[["Year", LGA_COL, RATE_COL]]], ignore_index=True
)

# Drop rows with no rate (Unincorporated Vic)
crime_all = crime_all.dropna(subset=[RATE_COL])
crime_all = crime_all.sort_values(["Year", LGA_COL]).reset_index(drop=True)

print("Years:", sorted(crime_all["Year"].unique()))
print("Rows:", len(crime_all))
print("LGAs:", crime_all[LGA_COL].nunique())
print("Duplicate Year/LGA:", crime_all.duplicated(["Year", LGA_COL]).sum())

# Save the COMBINED table
output_file = OUTPUT_DIR / "vic_crime_by_lga.csv"
crime_all.to_csv(output_file, index=False)
print("Saved to", output_file.resolve())

Years: [np.int64(2012), np.int64(2013), np.int64(2014), np.int64(2015), np.int64(2016), np.int64(2017), np.int64(2018), np.int64(2019), np.int64(2020), np.int64(2021), np.int64(2022), np.int64(2023), np.int64(2024), np.int64(2025), np.int64(2026)]
Rows: 1185
LGAs: 79
Duplicate Year/LGA: 0
Saved to /Users/markmchaileh/Documents/Applied Data Science/industry-project-group-1/external_data/vic_crime_by_lga.csv
